# Fingrid (Finland) – Bids and XML Deserialization

This notebook covers bid submission to Fingrid, the Finnish TSO, and demonstrates
the `deserialize_reserve_bid_document()` function for parsing incoming CIM XML.

| Parameter | Value |
|---|---|
| Receiver EIC | `10X1001A1001A264` |
| Control area EIC | `10YFI-1--------U` |
| Minimum bid | 1 MW |
| Max bids per message | **2000** (stricter than the common 4000 limit) |
| Heartbeat | Not required |
| Resource coding scheme | A01 (EIC) |
| Inclusive bids | Supported (aggregated resources) |
| Period shift | Not supported |
| Voluntary bid ID | Via `Reason` element, code A95, max 100 chars |
| Conditional links | Supported, including for inclusive bids |
| Product type change | A05 ↔ A07 allowed on update |
| Activation response | BSP accountable regardless of response |
| BEGOT | 30 days |

## What this notebook covers

1. Simple divisible and indivisible bids to Fingrid
2. Inclusive bid group (aggregated resources — Fingrid-specific)
3. Validating the 2000-bid-per-message limit
4. XML serialize → deserialize round-trip
5. Namespace flexibility: the deserializer accepts both the NBM and IEC namespace URIs

## Imports

In [1]:
from nexa_mfrr_eam import (
    Bid,
    BidDocument,
    BidDocumentModel,
    BiddingZone,
    Direction,
    InclusiveGroup,
    MARIMode,
    MarketProductType,
    TSO,
    deserialize_reserve_bid_document,
)

# Synthetic BSP party ID (EIC)
BSP_PARTY_ID = "10XBSP-FI-001---A"
BSP_CODING_SCHEME = "A01"

# Synthetic resource objects (EIC — Fingrid uses A01)
HYDRO_EIC = "10XHYDRO-FI-001-A"
WIND_EIC_1 = "10XWIND-FI-001--A"
WIND_EIC_2 = "10XWIND-FI-002--A"
WIND_EIC_3 = "10XWIND-FI-003--A"

## Part 1: Simple Bids to Fingrid

Fingrid uses EIC (`A01`) as the resource coding scheme.  The minimum bid is 1 MW
(same as the common Nordic minimum).  Both divisible and indivisible bids are
supported.

In [2]:
# Divisible up-regulation bid
bid_div = (
    Bid.up(volume_mw=40, price_eur=68.00)
    .divisible(min_volume_mw=10)
    .for_mtu("2026-03-21T10:00Z")
    .resource(HYDRO_EIC, coding_scheme="A01")
    .bidding_zone(BiddingZone.FI)
    .product_type(MarketProductType.SCHEDULED_AND_DIRECT)  # A07
    .build()
)

# Indivisible down-regulation bid
bid_indivis = (
    Bid.down(volume_mw=20, price_eur=55.00)
    .indivisible()
    .for_mtu("2026-03-21T10:00Z")
    .resource(HYDRO_EIC, coding_scheme="A01")
    .bidding_zone(BiddingZone.FI)
    .product_type(MarketProductType.SCHEDULED_ONLY)  # A05
    .build()
)

doc = (
    BidDocument(tso=TSO.FINGRID)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bid(bid_div)
    .add_bid(bid_indivis)
    .build()
)

errors = doc.validate(mari_mode=MARIMode.PRE_MARI)
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print(f"Document valid — {doc.time_series_count} bids")
    print(f"Receiver EIC:  {doc.model.receiver_mrid}")
    print(f"Domain EIC:    {doc.model.domain_mrid}")
    print(f"Period:        {doc.model.reserve_bid_period_start.strftime('%H:%MZ')} "
          f"to {doc.model.reserve_bid_period_end.strftime('%H:%MZ')}")

Document valid — 2 bids
Receiver EIC:  10X1001A1001A264
Domain EIC:    10YFI-1--------U
Period:        10:00Z to 10:15Z


## Part 2: Inclusive Bid Group (Aggregated Resources)

Fingrid supports **inclusive bid groups** for aggregating multiple physical
resources that must be activated together in the same proportion.  If one
component is activated, all components are activated at the same fractional
level.

### Inclusive group rules

- All components must share the **same price**, direction, product type, period,
  and bidding zone.
- Cannot be combined with exclusive or multipart groups.
- Supported by Fingrid and Statnett; **not** supported by Energinet or SVK.
- Partial activation rounds to the next full MW (or 0.1 MW for aggregated bids
  at Fingrid).

In [3]:
# Three wind turbines aggregated into one inclusive group.
# InclusiveGroup uses .add_component() — components share direction/MTU/zone
# set at the group level; per-component resource overrides are supported.
group = (
    InclusiveGroup(bidding_zone=BiddingZone.FI)
    .direction(Direction.UP)
    .for_mtu("2026-03-21T10:00Z")
    .product_type(MarketProductType.SCHEDULED_AND_DIRECT)
    .add_component(volume_mw=15, price_eur=72.00, min_volume_mw=1,
                   resource_id=WIND_EIC_1, resource_coding_scheme="A01")
    .add_component(volume_mw=20, price_eur=72.00, min_volume_mw=1,
                   resource_id=WIND_EIC_2, resource_coding_scheme="A01")
    .add_component(volume_mw=10, price_eur=72.00, min_volume_mw=1,
                   resource_id=WIND_EIC_3, resource_coding_scheme="A01")
    .build()
)

doc_group = (
    BidDocument(tso=TSO.FINGRID)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bids(group)
    .build()
)

errors = doc_group.validate(mari_mode=MARIMode.PRE_MARI)
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    # All three share the same inclusiveBidsIdentification UUID
    inclusive_id = doc_group.model.bid_time_series[0].inclusive_bids_identification
    all_same = all(
        b.inclusive_bids_identification == inclusive_id
        for b in doc_group.model.bid_time_series
    )
    print(f"Inclusive group ID: {inclusive_id}")
    print(f"All components share ID: {all_same}")
    print(f"Total volume: {sum(int(b.period.point.quantity) for b in doc_group.model.bid_time_series)} MW")

Inclusive group ID: 1df5f3ab-9405-45e5-b00c-64165f595624
All components share ID: True
Total volume: 45 MW


## Part 3: The 2000-Bid-Per-Message Limit

Fingrid enforces a **2000 bid maximum per message**, stricter than the common
Nordic 4000-bid limit.  Document validation returns an error if you exceed this.

In [4]:
from datetime import datetime, timezone, timedelta

# Build 2001 bids across consecutive MTUs to trip the limit
many_bids = []
base_mtu = datetime(2026, 3, 21, 0, 0, tzinfo=timezone.utc)
for i in range(2001):
    mtu = base_mtu + timedelta(minutes=15 * (i % 96))
    many_bids.append(
        Bid.up(volume_mw=5, price_eur=60.00)
        .divisible(min_volume_mw=1)
        .for_mtu(mtu)
        .resource(HYDRO_EIC, coding_scheme="A01")
        .bidding_zone(BiddingZone.FI)
        .product_type(MarketProductType.SCHEDULED_ONLY)
        .build()
    )

doc_too_many = (
    BidDocument(tso=TSO.FINGRID)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bids(many_bids)
    .build()
)

errors = doc_too_many.validate(mari_mode=MARIMode.PRE_MARI)
print(f"Bids in document: {doc_too_many.time_series_count}")
print("Validation errors (expected):")
for e in errors:
    print(f"  {e}")

Bids in document: 2001
Validation errors (expected):
  Document contains 2001 BidTimeSeries which exceeds the TSO limit of 2000


## Part 4: XML Serialize → Deserialize Round-Trip

`deserialize_reserve_bid_document()` parses a `ReserveBid_MarketDocument` XML
byte string back to a `BidDocumentModel`.  This is the inverse of `doc.to_xml()`.

Use cases:
- Verifying what you sent matches what the TSO received (before ACK arrives)
- Parsing stored/archived XML messages for audit or reconciliation
- Round-trip testing your own serialization logic

In [5]:
# Build a document with two bids
bid_a = (
    Bid.up(volume_mw=30, price_eur=75.00)
    .divisible(min_volume_mw=5)
    .for_mtu("2026-03-21T10:00Z")
    .resource(HYDRO_EIC, coding_scheme="A01")
    .bidding_zone(BiddingZone.FI)
    .product_type(MarketProductType.SCHEDULED_AND_DIRECT)
    .build()
)

bid_b = (
    Bid.up(volume_mw=20, price_eur=90.00)
    .indivisible()
    .for_mtu("2026-03-21T10:15Z")
    .resource(HYDRO_EIC, coding_scheme="A01")
    .bidding_zone(BiddingZone.FI)
    .product_type(MarketProductType.SCHEDULED_ONLY)
    .build()
)

built_doc = (
    BidDocument(tso=TSO.FINGRID)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bid(bid_a)
    .add_bid(bid_b)
    .build()
)

# Serialize to XML
xml_bytes: bytes = built_doc.to_xml()
print(f"Serialized XML: {len(xml_bytes)} bytes")

# Deserialize back to a model
parsed: BidDocumentModel = deserialize_reserve_bid_document(xml_bytes)

print(f"\nParsed document mRID:  {parsed.mrid}")
print(f"Original document mRID: {built_doc.model.mrid}")
print(f"Round-trip mRID match:  {parsed.mrid == built_doc.model.mrid}")

print(f"\nBid time series count: {len(parsed.bid_time_series)}")
for bts in parsed.bid_time_series:
    mtu_start = bts.period.time_interval_start.strftime("%H:%MZ")
    print(f"  {mtu_start}  {bts.period.point.quantity} MW @ "
          f"{bts.period.point.energy_price} EUR/MWh  "
          f"({'divisible' if bts.divisible_code == 'A01' else 'indivisible'})")

Serialized XML: 3699 bytes

Parsed document mRID:  c60c7356-9d9c-4bc1-8358-05e65bf34872
Original document mRID: c60c7356-9d9c-4bc1-8358-05e65bf34872
Round-trip mRID match:  True

Bid time series count: 2
  10:00Z  30 MW @ 75.0 EUR/MWh  (divisible)
  10:15Z  20 MW @ 90.0 EUR/MWh  (indivisible)


### Full field comparison

The deserialized model should be identical to the original.  Check a few key
fields across both bid time series.

In [6]:
orig_bids = built_doc.model.bid_time_series
parsed_bids = parsed.bid_time_series

for i, (orig, parsed_b) in enumerate(zip(orig_bids, parsed_bids)):
    print(f"Bid {i + 1}:")
    print(f"  mRID match:              {orig.mrid == parsed_b.mrid}")
    print(f"  flow direction match:    {orig.flow_direction == parsed_b.flow_direction}")
    print(f"  divisible code match:    {orig.divisible_code == parsed_b.divisible_code}")
    print(f"  resource mRID match:     {orig.registered_resource_mrid == parsed_b.registered_resource_mrid}")
    print(f"  coding scheme match:     {orig.registered_resource_coding_scheme == parsed_b.registered_resource_coding_scheme}")
    print(f"  quantity match:          {orig.period.point.quantity == parsed_b.period.point.quantity}")
    print(f"  price match:             {orig.period.point.energy_price == parsed_b.period.point.energy_price}")
    print(f"  MTU start match:         {orig.period.time_interval_start == parsed_b.period.time_interval_start}")
    print(f"  market product match:    {orig.standard_market_product_type == parsed_b.standard_market_product_type}")

Bid 1:
  mRID match:              True
  flow direction match:    True
  divisible code match:    True
  resource mRID match:     True
  coding scheme match:     True
  quantity match:          True
  price match:             True
  MTU start match:         True
  market product match:    True
Bid 2:
  mRID match:              True
  flow direction match:    True
  divisible code match:    True
  resource mRID match:     True
  coding scheme match:     True
  quantity match:          True
  price match:             True
  MTU start match:         True
  market product match:    True


## Part 5: Namespace Flexibility

Two namespace URIs are in use across the Nordic mFRR EAM ecosystem:

| Namespace | Used by |
|---|---|
| `urn:iec62325.351:tc57wg16:451-7:reservebiddocument:7:2` | Default serialization output; Statnett example XML |
| `urn:iec62325:ediel:nbm:reservebiddocument:7:2` | Vendored NBM XSD |

The deserializer accepts both.  Here we swap the namespace in the raw XML bytes
and confirm parsing still works.

In [7]:
from nexa_mfrr_eam.xml.namespaces import IEC_NAMESPACE, NBM_NAMESPACE

print(f"IEC namespace (default): {IEC_NAMESPACE}")
print(f"NBM namespace:           {NBM_NAMESPACE}")

# The default serialization uses the IEC namespace
assert IEC_NAMESPACE.encode() in xml_bytes, "IEC namespace not found in serialized XML"
print(f"\nSerialized with IEC namespace: True")

# Swap namespace to NBM and verify deserialization still works
xml_nbm = xml_bytes.replace(IEC_NAMESPACE.encode(), NBM_NAMESPACE.encode())
parsed_nbm: BidDocumentModel = deserialize_reserve_bid_document(xml_nbm)

print(f"Parsed with NBM namespace:     True")
print(f"mRID preserved:                {parsed_nbm.mrid == parsed.mrid}")
print(f"Bid count preserved:           {len(parsed_nbm.bid_time_series) == len(parsed.bid_time_series)}")

IEC namespace (default): urn:iec62325.351:tc57wg16:451-7:reservebiddocument:7:2
NBM namespace:           urn:iec62325:ediel:nbm:reservebiddocument:7:2

Serialized with IEC namespace: True
Parsed with NBM namespace:     True
mRID preserved:                True
Bid count preserved:           True


## Summary

| Feature | How to use |
|---|---|
| Submit to Fingrid | `BidDocument(tso=TSO.FINGRID)` — receiver EIC `10X1001A1001A264` |
| Resource coding | EIC (`A01`) — use your registered EIC resource IDs |
| Inclusive group | `InclusiveGroup(bidding_zone=...).add_component(volume_mw, price_eur, resource_id=...).build()` |
| Max bids check | `doc.validate()` returns an error if > 2000 bids |
| Serialize | `doc.to_xml()` → `bytes` |
| Deserialize | `deserialize_reserve_bid_document(xml_bytes)` → `BidDocumentModel` |
| Namespace | Deserializer accepts both NBM (`ediel:nbm`) and IEC (`tc57wg16`) URIs |

**What is NOT supported for Fingrid:**
- Period shift
- Heartbeat
- Non-standard bids (mFRR-D, overbelastning)
- Production type field (`mktPSRType.psrType`) — this is Denmark-only